# 06_bt_pairs_new — BT Pair Schedule Generation (hub-spoke architectuur)

Bouwt een pairwise comparison schedule voor Bradley-Terry (BT) schatting
via een gelaagd **hub-spoke ontwerp** dat graph-connectivity garandeert ongeacht uitval van raters.

## Architectuur

| Laag | Naam | Doel |
|------|------|------|
| 1 | **Backbone (hub-hub)** | Connectivity *tussen* alle 6 strata via spanning tree + diagonalen. Hubs gekozen op verkeersintensiteit (p33 en p67 per stratum). |
| 2a | **Hub-spoke (intra-stratum)** | Elke spoke gekoppeld aan één hub in hetzelfde stratum — elke node krijgt minstens één vergelijking. |
| 2b | **Diagonale spoke-spoke** | Directe cross-strata verificatie tussen spokes die op zowel type als prioriteit van elkaar verschillen. |
| 3 | **Within-stratum spoke-spoke** | Lokale dekking met resterend budget; gesorteerd op diversiteitsscore (intensiteitsverschil × centrumvlag). |

## Connectivity-garantie

De backbone (laag 1) is fully connected over alle hubs. Laag 2a koppelt elke spoke aan die backbone.
De complete graph is daarmee connected — BT-schatting kan op alle nodes worden uitgevoerd.

## Input / Output

**Input:** `data/processed/sampled_legs_directional.csv` (368 rijen, notebook 05)

**Output:**
- `data/processed/survey_design_v2.csv` — masterbestand survey tool (één rij per rater × taak)
- `data/processed/rater_schedules_v2/` — per-rater CSV's
- `data/processed/bt_design_report_v2.txt` — diagnostisch rapport

**Afhankelijkheid:** notebook 05 moet eerst zijn uitgevoerd.

## 0. Imports & configuratie

In [157]:
import os
import random
import itertools
from collections import defaultdict, Counter

import pandas as pd
import numpy as np
import networkx as nx

# --- Paden ---
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections"
INPUT_PATH         = os.path.join(PROJECT_DIR, "data", "processed", "sampled_legs_directional_clean.csv")
OUTPUT_DESIGN_PATH = os.path.join(PROJECT_DIR, "data", "processed", "survey_design_v2.csv")
SUMMARY_PATH       = os.path.join(PROJECT_DIR, "data", "processed", "bt_design_report_v2.txt")
RATER_SCHEDULE_DIR = os.path.join(PROJECT_DIR, "data", "processed", "rater_schedules_v2")
OUTPUT_DIR         = os.path.join(PROJECT_DIR, "output")

# --- Reproducibiliteit ---
RANDOM_SEED = 42

# --- Populatie & steekproef ---
# Doelgrootte van de gelabelde subset. Past binnen het raterbudget:
# 14 raters × 40 pairs = 560 judgments → ~3-4 vergelijkingen per node bij 150 nodes.
N_INTERSECTIONS_TARGET = 150

# --- Raterbudget ---
N_RATERS        = 14
PAIRS_PER_RATER = 40
N_ANCHOR_PAIRS  = 7   # ankerpairs: gezien door ALLE raters, voor inter-rater betrouwbaarheid

# --- Hub configuratie ---
# Aantal hubs per stratum. T_VRI is klein (18 nodes) → slechts 1 hub.
HUBS_PER_STRATUM = {
    "4+_VRI":           2,
    "4+_geen_voorrang": 2,
    "4+_voorrang":      2,
    "T_VRI":            1,
    "T_geen_voorrang":  2,
    "T_voorrang":       2,
}
# Totaal hubs: 11

# --- Replicatie per laag ---
# Hoe vaak elke pair wordt vertoond aan verschillende raters.
REPLICATION = {
    "backbone":  6,   # Layer 1: hoge replicatie — ruggengraat van de graph
    "hub_spoke": 2,   # Layer 2a
    "diagonal":  2,   # Layer 2b
    "within":    2,   # Layer 3
}

# --- Spanning tree over de 6 strata (5 verbindingen = minimum voor connectivity) ---
SPANNING_TREE_LINKS = [
    ("4+_VRI",           "T_VRI"),
    ("4+_geen_voorrang", "T_geen_voorrang"),
    ("4+_voorrang",      "T_voorrang"),
    ("4+_VRI",           "4+_geen_voorrang"),
    ("4+_geen_voorrang", "4+_voorrang"),
]

# --- Diagonale verbindingen (3 extra cross-strata links voor robuustheid) ---
DIAGONAL_LINKS = [
    ("4+_VRI",           "T_geen_voorrang"),
    ("4+_geen_voorrang", "T_VRI"),
    ("4+_voorrang",      "T_VRI"),
]

print("Configuratie geladen.")
print(f"  Doel: {N_INTERSECTIONS_TARGET} intersecties")
print(f"  Budget: {N_RATERS} raters × {PAIRS_PER_RATER} pairs = {N_RATERS * PAIRS_PER_RATER} judgments")

Configuratie geladen.
  Doel: 150 intersecties
  Budget: 14 raters × 40 pairs = 560 judgments


## 1. Data laden & subsamplen naar 150 intersecties

De volledige populatie bestaat uit 368 intersecties (notebook 05). We samplen proportioneel terug
naar 150 zodat het raterbudget (14 × 40 = 560 judgments) circa 3-4 vergelijkingen per node oplevert.

T_VRI (18 nodes) is te klein om verder te verkleinen en wordt volledig meegenomen.

In [158]:
rng = random.Random(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

df = pd.read_csv(INPUT_PATH)
print(f"Geladen: {len(df)} rijen uit {os.path.basename(INPUT_PATH)}")

# Bouw stratum-label uit dim_type en dim_priority
df["stratum"] = df["dim_type"].str.strip() + "_" + df["dim_priority"].str.strip()

# Vul ontbrekende intensiteit op met mediaan
if "intensity_wvk" not in df.columns:
    df["intensity_wvk"] = np.nan
df["intensity_wvk"] = df["intensity_wvk"].fillna(df["intensity_wvk"].median())

# Centrum flag — fall back naar False als kolom ontbreekt
if "is_centrum" not in df.columns:
    df["is_centrum"] = False
df["is_centrum"] = df["is_centrum"].fillna(False).astype(bool)

# Intensiteitsklasse op basis van p33/p67, bepaald over de volledige 368-node populatie
# zodat de grenzen consistent zijn ongeacht de subsample verderop.
# retbins=True geeft de numerieke grenswaarden terug voor rapportage.
df["intensity_q"], intensity_bins = pd.qcut(
    df["intensity_wvk"],
    q=[0, 0.33, 0.67, 1.0],
    labels=["Laag", "Middelmatig", "Hoog"],
    duplicates="drop",
    retbins=True,
)

print(f"\nIntensiteitsdrempelwaarden (P33/P67, volledige populatie van {len(df)} nodes):")
print(f"  Laag        : < {intensity_bins[1]:,.0f} vrtg/dag")
print(f"  Middelmatig : {intensity_bins[1]:,.0f} – {intensity_bins[2]:,.0f} vrtg/dag")
print(f"  Hoog        : > {intensity_bins[2]:,.0f} vrtg/dag")

print(f"\nVerdeling intensiteitsklasse (p33/p67 op volledige populatie):")
print(df["intensity_q"].value_counts().sort_index().to_string())

# --- Proportionele subsample naar N_INTERSECTIONS_TARGET ---
# Elk stratum levert een fractie evenredig aan zijn aandeel in de totale populatie.
stratum_counts = df["stratum"].value_counts()
total  = len(df)
target = N_INTERSECTIONS_TARGET

sampled_dfs = []
for stratum, count in stratum_counts.items():
    group    = df[df["stratum"] == stratum].copy()
    n_sample = max(1, round(count / total * target))
    n_sample = min(n_sample, len(group))   # nooit meer samplen dan beschikbaar
    sampled_dfs.append(group.sample(n=n_sample, random_state=RANDOM_SEED))

inter_df = pd.concat(sampled_dfs).reset_index(drop=True)
# intensity_q is already on each row from the full-population computation above;
# no recalculation needed — consistent thresholds across full and subsampled set.

print(f"\nSubsample: {len(inter_df)} intersecties over strata:")
for s, g in inter_df.groupby("stratum"):
    print(f"  {s}: {len(g)} intersecties")

# Metadata-index voor snelle lookups in hulpfuncties
meta          = inter_df.set_index("intersection_id")
stratum_nodes = {s: g["intersection_id"].tolist() for s, g in inter_df.groupby("stratum")}

Geladen: 329 rijen uit sampled_legs_directional_clean.csv

Intensiteitsdrempelwaarden (P33/P67, volledige populatie van 329 nodes):
  Laag        : < 1,000 vrtg/dag
  Middelmatig : 1,000 – 5,838 vrtg/dag
  Hoog        : > 5,838 vrtg/dag

Verdeling intensiteitsklasse (p33/p67 op volledige populatie):
intensity_q
Laag           113
Middelmatig    107
Hoog           109

Subsample: 149 intersecties over strata:
  4+_VRI: 31 intersecties
  4+_geen_voorrang: 30 intersecties
  4+_voorrang: 31 intersecties
  T_VRI: 5 intersecties
  T_geen_voorrang: 28 intersecties
  T_voorrang: 24 intersecties


## 2. Hubselectie

Per stratum worden 1 of 2 representatieve **hubs** gekozen:

- **Hub A** — node dichtstbijzijnde het 33e percentiel van `intensity_wvk` in het stratum
- **Hub B** — node dichtstbijzijnde het 67e percentiel (altijd een andere node dan hub A)

Dit garandeert spreiding over de intensiteitsdimensie. De hubs kunnen hieronder handmatig worden
overschreven door de `intersection_id` in te vullen.

In [159]:
hubs = {}   # stratum -> list van hub intersection_ids

for stratum, nodes in stratum_nodes.items():
    n_hubs      = HUBS_PER_STRATUM[stratum]
    intensities = meta.loc[nodes, "intensity_wvk"]

    if n_hubs == 1:
        # Kies node dichtstbijzijnde mediaan (enige hub voor kleine strata)
        hub_a = (intensities - intensities.median()).abs().idxmin()
        hubs[stratum] = [hub_a]
    else:
        # Hub A: dichtst bij 33e percentiel
        hub_a = (intensities - intensities.quantile(0.33)).abs().idxmin()
        # Hub B: dichtst bij 67e percentiel, niet dezelfde als hub A
        remaining = [n for n in nodes if n != hub_a]
        hub_b = (intensities[remaining] - intensities.quantile(0.67)).abs().idxmin()
        hubs[stratum] = [hub_a, hub_b]

print("Geselecteerde hubs:")
for stratum, hub_ids in hubs.items():
    for h in hub_ids:
        print(f"  {stratum}: node {h}  (intensity_wvk={meta.loc[h, 'intensity_wvk']:.0f})")

# --- HANDMATIGE OVERRIDE (optioneel) ---
# Verwijder de # om een stratum handmatig te overschrijven met eigen intersection_id's.
# hubs["4+_VRI"] = [12345, 67890]

# Verzamel alle hub-IDs in een set voor snelle lookups
all_hub_ids  = set(h for hub_list in hubs.values() for h in hub_list)
spokes       = {s: [n for n in nodes if n not in all_hub_ids] for s, nodes in stratum_nodes.items()}
total_spokes = sum(len(v) for v in spokes.values())
print(f"\nTotaal hubs: {len(all_hub_ids)} | Totaal spokes: {total_spokes}")

Geselecteerde hubs:
  4+_VRI: node 181265038  (intensity_wvk=6800)
  4+_VRI: node 183272020  (intensity_wvk=9100)
  4+_geen_voorrang: node 600483010  (intensity_wvk=350)
  4+_geen_voorrang: node 182270019  (intensity_wvk=1300)
  4+_voorrang: node 183277002  (intensity_wvk=1100)
  4+_voorrang: node 181277036  (intensity_wvk=5200)
  T_VRI: node 176275195  (intensity_wvk=6450)
  T_geen_voorrang: node 178273039  (intensity_wvk=400)
  T_geen_voorrang: node 183263020  (intensity_wvk=1200)
  T_voorrang: node 184273072  (intensity_wvk=2725)
  T_voorrang: node 600488624  (intensity_wvk=9100)

Totaal hubs: 11 | Totaal spokes: 138


## 3. Hulpfuncties

Drie kleine functies die in meerdere secties worden hergebruikt:

- `make_pair` — canonieke paarrepresentatie (laagste ID eerst) zodat `frozenset`-deduplicatie werkt
- `intensity_spread` / `centrum_spread` — attribuutdiversiteit per pair
- `diversity_score` — gecombineerde score; hogere waarde = informatiever pair voor BT-schatting

In [160]:
def make_pair(a, b):
    # Canonieke representatie: kleinste ID eerst
    return (min(a, b), max(a, b))

# Ordinale mapping voor de drie intensiteitsklassen — nodig voor het berekenen van het verschil
_INTENSITY_ORD = {"laag": 0, "midden": 1, "hoog": 2}

def intensity_spread(a, b):
    # Absolute verschil in intensiteitsklasse (0–2 schaal: laag=0, midden=1, hoog=2)
    return abs(
        _INTENSITY_ORD.get(meta.loc[a, "intensity_q"], 0) -
        _INTENSITY_ORD.get(meta.loc[b, "intensity_q"], 0)
    )

def centrum_spread(a, b):
    # 1 als de twee nodes verschillen in centrum-status, anders 0
    return int(bool(meta.loc[a, "is_centrum"]) != bool(meta.loc[b, "is_centrum"]))

def diversity_score(a, b):
    # Hogere score = informatiever pair voor BT-schatting
    return intensity_spread(a, b) * 2 + centrum_spread(a, b)

print("Hulpfuncties gedefinieerd.")

Hulpfuncties gedefinieerd.


## 4. Layer 1 — Backbone (hub-hub pairs)

De backbone bestaat uit drie componenten:

- **4a. Intra-stratum hub pairs** — de twee hubs binnen hetzelfde stratum worden direct vergeleken
- **4b. Spanning tree inter-strata** — 5 links verbinden alle 6 strata tot één samenhangende component
- **4c. Diagonale hub-hub links** — 3 extra verbindingen voor robuustheid (cross-arm én cross-prioriteit)

Per inter-stratum link wordt het meest diverse hub-paar gekozen (hoogste `diversity_score`).

In [161]:
backbone_pairs = set()

# 4a. Intra-stratum hub pairs: hubs binnen hetzelfde stratum direct vergelijken
print("Layer 1a — Intra-stratum hub pairs:")
for stratum, hub_list in hubs.items():
    for a, b in itertools.combinations(hub_list, 2):
        backbone_pairs.add(make_pair(a, b))
        print(f"  {stratum}: {a} <-> {b}")

# 4b. Spanning tree: minimale set links voor cross-strata connectivity
print("\nLayer 1b — Spanning tree inter-strata:")
for s_a, s_b in SPANNING_TREE_LINKS:
    # Kies het meest diverse hub-paar over de twee strata
    candidates = list(itertools.product(hubs[s_a], hubs[s_b]))
    best = max(candidates, key=lambda p: diversity_score(p[0], p[1]))
    backbone_pairs.add(make_pair(best[0], best[1]))
    print(f"  {s_a} <-> {s_b}: node {best[0]} <-> {best[1]}")

# 4c. Diagonale links: extra verbindingen die arm-type en prioriteit tegelijk overspannen
print("\nLayer 1c — Diagonale hub-hub links:")
for s_a, s_b in DIAGONAL_LINKS:
    candidates = list(itertools.product(hubs[s_a], hubs[s_b]))
    best = max(candidates, key=lambda p: diversity_score(p[0], p[1]))
    backbone_pairs.add(make_pair(best[0], best[1]))
    print(f"  {s_a} <-> {s_b}: node {best[0]} <-> {best[1]}")

backbone_pairs = list(backbone_pairs)
print(f"\nLayer 1 totaal: {len(backbone_pairs)} unieke hub-hub pairs")

Layer 1a — Intra-stratum hub pairs:
  4+_VRI: 181265038 <-> 183272020
  4+_geen_voorrang: 600483010 <-> 182270019
  4+_voorrang: 183277002 <-> 181277036
  T_geen_voorrang: 178273039 <-> 183263020
  T_voorrang: 184273072 <-> 600488624

Layer 1b — Spanning tree inter-strata:
  4+_VRI <-> T_VRI: node 183272020 <-> 176275195
  4+_geen_voorrang <-> T_geen_voorrang: node 600483010 <-> 178273039
  4+_voorrang <-> T_voorrang: node 183277002 <-> 184273072
  4+_VRI <-> 4+_geen_voorrang: node 183272020 <-> 600483010
  4+_geen_voorrang <-> 4+_voorrang: node 600483010 <-> 183277002

Layer 1c — Diagonale hub-hub links:
  4+_VRI <-> T_geen_voorrang: node 183272020 <-> 178273039
  4+_geen_voorrang <-> T_VRI: node 600483010 <-> 176275195
  4+_voorrang <-> T_VRI: node 183277002 <-> 176275195

Layer 1 totaal: 13 unieke hub-hub pairs


## 5. Layer 2a — Hub-spoke pairs

Elke spoke wordt vergeleken met precies één hub in hetzelfde stratum:

- Bij strata met **1 hub**: alle spokes gaan naar die ene hub
- Bij strata met **2 hubs**: spokes gesorteerd op intensiteit, afwisselend hub A en hub B

Dit legt alle spokes vast op de backbone en garandeert dat elke node minstens één vergelijking heeft.

In [162]:
hub_spoke_pairs = []

print("Layer 2a — Hub-spoke pairs:")
for stratum, spoke_list in spokes.items():
    hub_list = hubs[stratum]

    if len(hub_list) == 1:
        # Alle spokes aan de enige hub koppelen
        for spoke in spoke_list:
            hub_spoke_pairs.append(make_pair(hub_list[0], spoke))
    else:
        # Sorteer spokes op intensiteit en wissel af tussen hub A en hub B
        spoke_sorted = sorted(spoke_list, key=lambda n: meta.loc[n, "intensity_wvk"])
        for i, spoke in enumerate(spoke_sorted):
            hub = hub_list[i % 2]
            hub_spoke_pairs.append(make_pair(hub, spoke))

    print(f"  {stratum}: {len(spoke_list)} spokes → {len(spoke_list)} hub-spoke pairs")

print(f"\nLayer 2a totaal: {len(hub_spoke_pairs)} hub-spoke pairs")

Layer 2a — Hub-spoke pairs:
  4+_VRI: 29 spokes → 29 hub-spoke pairs
  4+_geen_voorrang: 28 spokes → 28 hub-spoke pairs
  4+_voorrang: 29 spokes → 29 hub-spoke pairs
  T_VRI: 4 spokes → 4 hub-spoke pairs
  T_geen_voorrang: 26 spokes → 26 hub-spoke pairs
  T_voorrang: 22 spokes → 22 hub-spoke pairs

Layer 2a totaal: 138 hub-spoke pairs


## 6. Layer 2b — Diagonale spoke-spoke pairs

Directe cross-strata vergelijkingen die *niet* via hubs lopen. Per link in `DIAGONAL_LINKS`
worden twee paren geselecteerd om de intensiteitsdimensie te overspannen:

- 1 paar: lage intensiteit uit stratum A vs. hoge intensiteit uit stratum B
- 1 paar: hoge intensiteit uit stratum A vs. lage intensiteit uit stratum B

In [163]:
diagonal_pairs = []
existing_pairs = set(map(frozenset, backbone_pairs + hub_spoke_pairs))

print("Layer 2b — Diagonale spoke-spoke pairs:")
for s_a, s_b in DIAGONAL_LINKS:
    spoke_a = spokes[s_a]
    spoke_b = spokes[s_b]

    if not spoke_a or not spoke_b:
        print(f"  {s_a} <-> {s_b}: geen spokes beschikbaar, overgeslagen")
        continue

    # Onderste en bovenste kwartiel per stratum voor representatieve uitersten
    low_a  = sorted(spoke_a, key=lambda n: meta.loc[n, "intensity_wvk"])[:max(1, len(spoke_a)//4)]
    high_a = sorted(spoke_a, key=lambda n: meta.loc[n, "intensity_wvk"])[-(max(1, len(spoke_a)//4)):]
    low_b  = sorted(spoke_b, key=lambda n: meta.loc[n, "intensity_wvk"])[:max(1, len(spoke_b)//4)]
    high_b = sorted(spoke_b, key=lambda n: meta.loc[n, "intensity_wvk"])[-(max(1, len(spoke_b)//4)):]

    added = 0
    for pool_a, pool_b in [(low_a, high_b), (high_a, low_b)]:
        for a in pool_a:
            for b in pool_b:
                p = make_pair(a, b)
                if frozenset(p) not in existing_pairs:
                    diagonal_pairs.append(p)
                    existing_pairs.add(frozenset(p))
                    added += 1
                    break
            if added >= 2:
                break
        if added >= 2:
            break

    print(f"  {s_a} <-> {s_b}: {added} diagonale spoke-spoke pairs")

print(f"\nLayer 2b totaal: {len(diagonal_pairs)} diagonale spoke-spoke pairs")

Layer 2b — Diagonale spoke-spoke pairs:
  4+_VRI <-> T_geen_voorrang: 2 diagonale spoke-spoke pairs
  4+_geen_voorrang <-> T_VRI: 2 diagonale spoke-spoke pairs
  4+_voorrang <-> T_VRI: 2 diagonale spoke-spoke pairs

Layer 2b totaal: 6 diagonale spoke-spoke pairs


## 7. Layer 3 — Within-stratum spoke-spoke pairs

Met het resterende raterbudget worden within-stratum spoke-spoke pairs toegevoegd.
Kandidaten worden gesorteerd op `diversity_score` zodat het beschikbare budget
optimaal wordt benut (meest informatieve pairs eerst).

In [164]:
# Bereken resterend budget na lagen 1, 2a en 2b
total_budget    = N_RATERS * (PAIRS_PER_RATER - N_ANCHOR_PAIRS)
backbone_slots  = len(backbone_pairs)  * REPLICATION["backbone"]
hub_spoke_slots = len(hub_spoke_pairs) * REPLICATION["hub_spoke"]
diagonal_slots  = len(diagonal_pairs)  * REPLICATION["diagonal"]
remaining_slots = total_budget - backbone_slots - hub_spoke_slots - diagonal_slots
n_within_select = max(0, remaining_slots // REPLICATION["within"])

print("Layer 3 — Within-stratum spoke-spoke pairs:")
print(f"  Totaal raterbudget (non-anchor): {total_budget}")
print(f"  Gebruikt door laag 1:            {backbone_slots}")
print(f"  Gebruikt door laag 2a:           {hub_spoke_slots}")
print(f"  Gebruikt door laag 2b:           {diagonal_slots}")
print(f"  Resterend: {remaining_slots} → selecteer {n_within_select} within-stratum pairs")

within_pairs = []

for stratum, spoke_list in spokes.items():
    if len(spoke_list) < 2:
        continue

    # Alle spoke-spoke kandidaten binnen dit stratum
    candidates = list(itertools.combinations(spoke_list, 2))

    # Sorteer op diversiteitsscore (hoog = meer informatie); random tiebreak voor reproduceerbaarheid
    candidates.sort(key=lambda p: (-diversity_score(p[0], p[1]), rng.random()))

    # Voeg toe zolang budget beschikbaar en pair nog niet bestaat
    for a, b in candidates:
        p = make_pair(a, b)
        if frozenset(p) not in existing_pairs and len(within_pairs) < n_within_select:
            within_pairs.append(p)
            existing_pairs.add(frozenset(p))

print(f"  Layer 3 totaal: {len(within_pairs)} within-stratum spoke-spoke pairs")

Layer 3 — Within-stratum spoke-spoke pairs:
  Totaal raterbudget (non-anchor): 462
  Gebruikt door laag 1:            78
  Gebruikt door laag 2a:           276
  Gebruikt door laag 2b:           12
  Resterend: 96 → selecteer 48 within-stratum pairs
  Layer 3 totaal: 48 within-stratum spoke-spoke pairs


## 8. Alle lagen samenvoegen & connectivity checken

Alle vier lagen worden gecombineerd tot één design. NetworkX verifieert of de graph connected is —
een noodzakelijke voorwaarde voor BT-schatting over alle nodes.

In [165]:
all_design_pairs = backbone_pairs + hub_spoke_pairs + diagonal_pairs + within_pairs

# Tag elk pair met zijn laag voor diagnostiek en output
pair_layer = {}
for p in backbone_pairs:  pair_layer[p] = "backbone"
for p in hub_spoke_pairs: pair_layer[p] = "hub_spoke"
for p in diagonal_pairs:  pair_layer[p] = "diagonal"
for p in within_pairs:    pair_layer[p] = "within"

# Bouw NetworkX graph voor connectiviteitscheck
all_node_ids = inter_df["intersection_id"].tolist()
G = nx.Graph()
G.add_nodes_from(all_node_ids)
G.add_edges_from(all_design_pairs)

connected      = nx.is_connected(G)
degree_series  = pd.Series(dict(G.degree()))

print(f"{'='*55}")
print(f"ONTWERP SAMENVATTING")
print(f"{'='*55}")
print(f"Nodes:          {len(all_node_ids)}")
print(f"Unieke pairs:   {len(all_design_pairs)}")
print(f"  Layer 1 backbone:    {len(backbone_pairs)}")
print(f"  Layer 2a hub-spoke:  {len(hub_spoke_pairs)}")
print(f"  Layer 2b diagonal:   {len(diagonal_pairs)}")
print(f"  Layer 3 within:      {len(within_pairs)}")
print(f"\nGraph connected: {connected}")
print(f"Graden per node:")
print(f"  Min:  {degree_series.min()}")
print(f"  Mean: {degree_series.mean():.1f}")
print(f"  Max:  {degree_series.max()}")
print(f"  Nodes met < 3 pairs: {(degree_series < 3).sum()}")

if not connected:
    n_comp = nx.number_connected_components(G)
    print(f"\nWARNING: graph niet connected — {n_comp} componenten!")
    print("Controleer of alle strata hubs hebben en spanning tree correct is geconfigureerd.")
else:
    print("\nOK — BT schatting kan worden uitgevoerd op alle nodes.")

ONTWERP SAMENVATTING
Nodes:          149
Unieke pairs:   205
  Layer 1 backbone:    13
  Layer 2a hub-spoke:  138
  Layer 2b diagonal:   6
  Layer 3 within:      48

Graph connected: True
Graden per node:
  Min:  1
  Mean: 2.8
  Max:  19
  Nodes met < 3 pairs: 112

OK — BT schatting kan worden uitgevoerd op alle nodes.


## 9. Raterblokken samenstellen

Elke rater krijgt een blok van `PAIRS_PER_RATER` pairs:
- **Ankerpairs** (`N_ANCHOR_PAIRS`) — gekozen uit backbone-pairs, gezien door **alle** raters
- **Unieke pairs** — overige pairs, load-balanced verdeeld over raters

Toewijzingsvolgorde: backbone → hub_spoke → diagonal → within.
Dit garandeert dat de ruggengraat prioriteit heeft bij onverwachte rateruitval.

In [166]:
# --- Ankerpairs selecteren uit backbone-pairs ---
rng_anchor = random.Random(RANDOM_SEED + 1)
anchor_pool = backbone_pairs.copy()
rng_anchor.shuffle(anchor_pool)
anchor_pairs = anchor_pool[:min(N_ANCHOR_PAIRS, len(anchor_pool))]
anchor_set   = set(anchor_pairs)
print(f"Ankerpairs geselecteerd: {len(anchor_pairs)}")

# Ankers worden aan ALLE raters gegeven. Pre-populate rater_seen_count (Counter per rater)
# zodat de load-balancer weet hoe vaak elk kruispunt al gezien is via ankerpairs.
# Counter ipv set: ook als een kruispunt al meerdere keren via ankers verschijnt, worden
# raters met de laagste blootstellingsfrequentie geprioriteerd boven raters met meer.
rater_seen_count = [Counter() for _ in range(N_RATERS)]
for a, b in anchor_pairs:
    for r in range(N_RATERS):
        rater_seen_count[r][a] += 1
        rater_seen_count[r][b] += 1

# --- Non-anchor pairs met replicatiegewicht ---
non_anchor_weighted = []
for p in backbone_pairs:
    if p not in anchor_set:
        non_anchor_weighted.append((p, REPLICATION["backbone"]))
for p in hub_spoke_pairs:
    non_anchor_weighted.append((p, REPLICATION["hub_spoke"]))
for p in diagonal_pairs:
    non_anchor_weighted.append((p, REPLICATION["diagonal"]))
for p in within_pairs:
    non_anchor_weighted.append((p, REPLICATION["within"]))

# Schud voor eerlijke verdeling
rng_assign = random.Random(RANDOM_SEED + 10)
rng_assign.shuffle(non_anchor_weighted)

# --- Load-balanced toewijzing met counter-gebaseerde deduplicatie ---
# Elk pair wordt aan precies R *verschillende* raters toegewezen.
# Sorteervolgorde: laagste max-blootstellingsfrequentie van de twee kruispunten bij die
# rater, dan minste totale belasting, dan random tiebreak. Dit differentieert ook tussen
# raters die een hub-kruispunt al 2x vs 3x gezien hebben — de set-aanpak behandelde
# beide gelijk omdat het kruispunt al "in de set" zat.
rater_load  = [0] * N_RATERS
rater_tasks = [[] for _ in range(N_RATERS)]

for pair, n_reps in non_anchor_weighted:
    a, b = pair
    priority = sorted(
        range(N_RATERS),
        key=lambda r: (max(rater_seen_count[r][a], rater_seen_count[r][b]), rater_load[r], rng_assign.random())
    )
    assigned = 0
    for r in priority:
        if assigned >= n_reps:
            break
        if rater_load[r] < (PAIRS_PER_RATER - N_ANCHOR_PAIRS):
            rater_tasks[r].append(pair)
            rater_load[r] += 1
            # Verhoog de teller voor beide kruispunten bij deze rater
            rater_seen_count[r][a] += 1
            rater_seen_count[r][b] += 1
            assigned += 1

# Trim naar minimum zodat alle sets gelijk zijn (vereist door de survey tool)
n_unique    = min(len(t) for t in rater_tasks)
rater_tasks = [t[:n_unique] for t in rater_tasks]

print(f"Non-anchor pairs per rater: {n_unique}")
print(f"Totaal per rater (incl. ankers): {n_unique + len(anchor_pairs)}")
print(f"Doelwaarde: {PAIRS_PER_RATER}")
if n_unique + len(anchor_pairs) != PAIRS_PER_RATER:
    print("  NOTE: verschil met doelwaarde — pas PAIRS_PER_RATER of replicaties aan.")

Ankerpairs geselecteerd: 7
Non-anchor pairs per rater: 29
Totaal per rater (incl. ankers): 36
Doelwaarde: 40
  NOTE: verschil met doelwaarde — pas PAIRS_PER_RATER of replicaties aan.


## 10. Output — survey_design_v2.csv & per-rater schedules

Bouwt de masterfile `survey_design_v2.csv` (één rij per rater × taak) en de individuele rater-CSV's.

Intensiteitswaarden worden omgezet van voertuigen/dag naar afgeronde voertuigen/uur (÷ 24, afgerond op 10)
voor weergave in de survey.

**Kwaliteitscheck:** alle sets moeten exact gelijk zijn van formaat (vereist door de survey tool).

In [167]:
def to_vph(x):
    # Voertuigen/dag → afgeronde voertuigen/uur (niet meer gebruikt voor output, behouden voor referentie)
    if pd.isna(x):
        return None
    return int(round(float(x) / 24 / 10) * 10)

# Bouw ankerpairs-dataframe (gedeeld door alle raters)
anchor_records = []
for p in anchor_pairs:
    a, b = p
    anchor_records.append({
        "pair_id":          frozenset(p).__hash__(),
        "intersection_a":   a,
        "image_path_a":     meta.loc[a, "photo_filepath"],
        "intensity_wvk_a":  meta.loc[a, "intensity_wvk"],
        "stratum_a":        meta.loc[a, "stratum"],
        "intersection_b":   b,
        "image_path_b":     meta.loc[b, "photo_filepath"],
        "intensity_wvk_b":  meta.loc[b, "intensity_wvk"],
        "stratum_b":        meta.loc[b, "stratum"],
        "is_cross_stratum": meta.loc[a, "stratum"] != meta.loc[b, "stratum"],
        "pair_type":        pair_layer.get(p, "backbone"),
        "is_anchor":        True,
    })
anchor_df = pd.DataFrame(anchor_records)

os.makedirs(RATER_SCHEDULE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_DESIGN_PATH), exist_ok=True)

design_rows = []

for i in range(N_RATERS):
    rater_id = f"rater_{i + 1:02d}"

    # Non-anchor slice voor deze rater
    unique_records = []
    for p in rater_tasks[i]:
        a, b = p
        unique_records.append({
            "pair_id":          frozenset(p).__hash__(),
            "intersection_a":   a,
            "image_path_a":     meta.loc[a, "photo_filepath"],
            "intensity_wvk_a":  meta.loc[a, "intensity_wvk"],
            "stratum_a":        meta.loc[a, "stratum"],
            "intersection_b":   b,
            "image_path_b":     meta.loc[b, "photo_filepath"],
            "intensity_wvk_b":  meta.loc[b, "intensity_wvk"],
            "stratum_b":        meta.loc[b, "stratum"],
            "is_cross_stratum": meta.loc[a, "stratum"] != meta.loc[b, "stratum"],
            "pair_type":        pair_layer.get(p, "within"),
            "is_anchor":        False,
        })
    unique_df = pd.DataFrame(unique_records)

    # Combineer ankers + unieke pairs en schud
    schedule = pd.concat([anchor_df, unique_df], ignore_index=True)
    schedule = schedule.sample(frac=1, random_state=RANDOM_SEED + i).reset_index(drop=True)
    schedule.to_csv(os.path.join(RATER_SCHEDULE_DIR, f"{rater_id}.csv"), index=False)

    # Voeg toe aan design_rows voor de master CSV
    for _, row in schedule.iterrows():
        design_rows.append({
            "set_id":                  i + 1,
            "pair_id":                 row["pair_id"],
            "is_anchor":               bool(row["is_anchor"]),
            "pair_type":               row["pair_type"],
            "is_cross_stratum":        bool(row["is_cross_stratum"]),
            "intersection_a":          row["intersection_a"],
            "intersection_b":          row["intersection_b"],
            "alt1_att1_streetview":    row["image_path_a"],
            # Intensiteitsklasse (laag/midden/hoog) uit de p33/p67-buckets berekend in cel 65b2513f
            "alt1_att2_intensiteit":   meta.loc[row["intersection_a"], "intensity_q"],
            "alt2_att1_streetview":    row["image_path_b"],
            "alt2_att2_intensiteit":   meta.loc[row["intersection_b"], "intensity_q"],
            "intensity_wvk_a":         row["intensity_wvk_a"],
            "intensity_wvk_b":         row["intensity_wvk_b"],
            "stratum_a":               row["stratum_a"],
            "stratum_b":               row["stratum_b"],
        })

design_df = pd.DataFrame(design_rows)
design_df.insert(0, "task_id", range(1, len(design_df) + 1))

# Geen Int64-cast nodig: alt*_att2_intensiteit zijn nu strings (laag/midden/hoog)

design_df.to_csv(OUTPUT_DESIGN_PATH, index=False)
design_df.to_csv(os.path.join(OUTPUT_DIR, "survey_design_v2.csv"), index=False)

# Kwaliteitscontrole set-sizes
set_sizes = design_df.groupby("set_id").size()
print(f"Saved survey_design_v2.csv: {len(design_df)} rijen | {N_RATERS} sets")
print(f"Set sizes: min={set_sizes.min()}, max={set_sizes.max()} (gelijk: {set_sizes.min() == set_sizes.max()})")
print(f"Per-rater schedules: {RATER_SCHEDULE_DIR}/")

Saved survey_design_v2.csv: 504 rijen | 14 sets
Set sizes: min=36, max=36 (gelijk: True)
Per-rater schedules: C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\rater_schedules_v2/


## 11. Diagnostisch rapport

Schrijft een tekstrapport met alle relevante designstatistieken naar `bt_design_report_v2.txt`,
inclusief de 10 nodes met de laagste graad (minste vergelijkingen) voor handmatige review.

In [168]:
report_lines = [
    "BT PAIR DESIGN RAPPORT — v2 (hub-spoke architectuur)",
    "=" * 60,
    f"Intersecties: {len(inter_df)}",
    f"Hubs: {len(all_hub_ids)}",
    f"Spokes: {total_spokes}",
    "",
    "PAIRS PER LAAG:",
    f"  Layer 1 backbone (hub-hub): {len(backbone_pairs)} pairs × {REPLICATION['backbone']} reps",
    f"  Layer 2a hub-spoke:         {len(hub_spoke_pairs)} pairs × {REPLICATION['hub_spoke']} reps",
    f"  Layer 2b diagonal:          {len(diagonal_pairs)} pairs × {REPLICATION['diagonal']} reps",
    f"  Layer 3 within-stratum:     {len(within_pairs)} pairs × {REPLICATION['within']} reps",
    f"  Totaal uniek:               {len(all_design_pairs)} pairs",
    "",
    "GRAAF:",
    f"  Connected: {connected}",
    f"  Min. graad: {degree_series.min()}",
    f"  Gem. graad: {degree_series.mean():.1f}",
    f"  Nodes met < 3 pairs: {(degree_series < 3).sum()}",
    "",
    "RATERS:",
    f"  {N_RATERS} raters × {PAIRS_PER_RATER} pairs = {N_RATERS * PAIRS_PER_RATER} totale judgments",
    f"  Ankerpairs (alle raters): {len(anchor_pairs)}",
    f"  Non-anchor per rater: {n_unique}",
    "",
    "GRADEN PER NODE (top-10 laagst gedekt):",
]

low_degree = degree_series.nsmallest(10)
for node_id, deg in low_degree.items():
    s      = meta.loc[node_id, "stratum"]
    is_hub = "HUB" if node_id in all_hub_ids else "spoke"
    report_lines.append(f"  node {node_id} ({s}, {is_hub}): {deg} vergelijkingen")

report_text = "\n".join(report_lines)
print(report_text)

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    f.write(report_text)
print(f"\nRapport opgeslagen: {SUMMARY_PATH}")

BT PAIR DESIGN RAPPORT — v2 (hub-spoke architectuur)
Intersecties: 149
Hubs: 11
Spokes: 138

PAIRS PER LAAG:
  Layer 1 backbone (hub-hub): 13 pairs × 6 reps
  Layer 2a hub-spoke:         138 pairs × 2 reps
  Layer 2b diagonal:          6 pairs × 2 reps
  Layer 3 within-stratum:     48 pairs × 2 reps
  Totaal uniek:               205 pairs

GRAAF:
  Connected: True
  Min. graad: 1
  Gem. graad: 2.8
  Nodes met < 3 pairs: 112

RATERS:
  14 raters × 40 pairs = 560 totale judgments
  Ankerpairs (alle raters): 7
  Non-anchor per rater: 29

GRADEN PER NODE (top-10 laagst gedekt):
  node 186275037 (4+_VRI, spoke): 1 vergelijkingen
  node 188274022 (4+_VRI, spoke): 1 vergelijkingen
  node 188266137 (4+_voorrang, spoke): 1 vergelijkingen
  node 179265077 (4+_voorrang, spoke): 1 vergelijkingen
  node 180272143 (4+_voorrang, spoke): 1 vergelijkingen
  node 183274028 (4+_voorrang, spoke): 1 vergelijkingen
  node 187265053 (4+_voorrang, spoke): 1 vergelijkingen
  node 192266023 (4+_voorrang, spoke)

## 12. survey.yaml bijwerken

Schrijft `tasks_per_respondent` terug naar de YAML-configuratie van de survey tool,
zodat die synchroon loopt met het gegenereerde design.

In [169]:
import yaml

SURVEY_YAML_PATH = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\core-usage\recipes\recipe-intersection-safety\survey.yaml"

tasks_per_respondent = len(anchor_pairs) + n_unique

with open(SURVEY_YAML_PATH, "r", encoding="utf-8") as f:
    survey = yaml.safe_load(f)

survey["survey"]["sections"]["body"]["activity_1"]["settings"]["tasks_per_respondent"] = tasks_per_respondent

with open(SURVEY_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(survey, f, allow_unicode=True, sort_keys=False)

print(f"survey.yaml bijgewerkt: tasks_per_respondent = {tasks_per_respondent}")

survey.yaml bijgewerkt: tasks_per_respondent = 36
